# Lección 5 — Clustering / aprendizaje no supervisado

**Tema:** un giro de paradigma: hasta ahora trabajamos con datos **etiquetados** (cada fila tenía un `y` que el modelo aprendía a predecir). En esta lección entramos al mundo del **aprendizaje no supervisado**, donde NO hay etiquetas y la pregunta cambia. Vamos a usar **K-means** como hilo conductor sobre casos concretos: segmentación de clientes, agrupamiento de viviendas, tipologías de tiendas. Veremos el algoritmo en cámara lenta, descubriremos por qué el **feature scaling** vuelve a ser clave, elegiremos el número de clusters con el **elbow method**, armaremos un **pipeline de clustering** análogo al supervisado, y cerraremos con una idea breve de **tokens** para conectar con el procesamiento de texto.

**Objetivos de esta lección:**
- Distinguir **aprendizaje no supervisado** (clustering) de **supervisado** (clasificación / regresión), y reconocer que **K-means NO es lo mismo que KNN**.
- Interpretar la salida visual de K-means en datasets 2D, viendo **paso a paso** cómo se mueven los centroides en cada iteración a partir de una inicialización mala.
- Reconocer visualmente que los algoritmos basados en **distancia** quedan dominados por la feature de mayor escala si no se aplica feature scaling.
- Aplicar el **elbow method** para elegir el número de clusters K cuando no se conoce a priori, y entender qué pasa si elegimos K más alto que el codo.
- Recorrer un **pipeline de clustering** completo con los 7 pasos del ciclo de un proyecto de ML, identificando qué cambia respecto al pipeline supervisado.
- Tener una primera idea de qué son los **tokens** — texto convertido a números para alimentar a un modelo.

**Side-note:** este notebook usa `numpy`, `pandas`, `matplotlib` y `scikit-learn`. Todas vienen preinstaladas en Google Colab — no hace falta `pip install`.

## Índice

1. **Hook: hasta ahora todo era supervisado** — el cambio de pregunta cuando no hay etiquetas.
2. **K-means con visualización paso a paso** — el algoritmo en cámara lenta sobre un caso de segmentación de clientes.
3. **Feature scaling en algoritmos de distancia** — por qué escalamos las features cuando las escalas son muy distintas.
4. **Elección de K + elbow method** — cómo elegir cuántos clusters cuando no sabés.
5. **Pipeline de clustering: los 7 pasos** — el ciclo de un proyecto de ML aplicado a clustering.
6. **Bonus: ¿qué es un token?** — texto convertido a números.

In [ ]:
# Setup — imports y configuración común a todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Estilo común para los plots
plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.grid'] = True

np.random.seed(42)

## Hook: hasta ahora todo era supervisado

Hasta acá, todo lo que vimos tenía **etiquetas**: cada fila del dataset tenía un `y` (la clase, el precio, el resultado) y el modelo aprendía a predecirlo. En la realidad, muchas veces no hay etiquetas — solo datos crudos.

**Caso concreto:** una empresa de retail nos pasa una tabla con 300 clientes y dos métricas por cliente: cuánto gasta por mes y con qué frecuencia visita la tienda. El equipo de marketing quiere armar **segmentos de clientes** para campañas distintas, pero NO viene con segmentos pre-definidos — quiere descubrirlos a partir de los datos.

Ahí la pregunta cambia: en lugar de *"¿qué clase es esto?"* pasa a ser *"¿qué grupos hay acá?"*. Eso es **clustering**, y es el reino del **aprendizaje no supervisado**.

In [ ]:
# Demo — el profesor ejecuta y explica
# 300 clientes con dos métricas. NO tenemos segmentos pre-definidos.
X_clientes, _ = make_blobs(
    n_samples=300,
    centers=[(3, 5), (9, 12), (14, 4)],
    cluster_std=1.2,
    random_state=42,
)
df_clientes = pd.DataFrame(X_clientes, columns=['gasto_mensual_miles', 'visitas_por_mes'])

print('Primeras filas del dataset de clientes:')
print(df_clientes.head())

plt.figure()
plt.scatter(
    df_clientes['gasto_mensual_miles'],
    df_clientes['visitas_por_mes'],
    s=20, color='gray', alpha=0.7,
)
plt.title('300 clientes — sin segmentos pre-definidos')
plt.xlabel('gasto mensual (miles de pesos)')
plt.ylabel('visitas por mes')
plt.show()

> **Aclaración crítica: K-means NO es lo mismo que KNN.**
>
> - **K-means** = **clustering** (no supervisado). Descubre grupos en datos SIN etiquetas.
> - **KNN** (K-Nearest Neighbors) = **clasificación** (supervisado). Predice una etiqueta basándose en los vecinos etiquetados más cercanos.
>
> Comparten la letra K y la idea de "distancia", pero resuelven problemas distintos. Si oís los dos nombres juntos, no los confundas.

## K-means con visualización paso a paso

Vamos a ver el algoritmo correr **en cámara lenta** sobre el dataset de clientes. Para que el movimiento de los centroides sea visible, **arrancamos con una inicialización deliberadamente mala**: los 3 centroides apilados cerca de una esquina. Cada frame muestra el estado tras un número creciente de iteraciones.

La idea de K-means es simple:

1. Elegir K centros iniciales (al azar, o como en este caso, mal a propósito).
2. **Asignar:** cada punto se etiqueta con el centro más cercano.
3. **Mover:** cada centro se reubica en el promedio de los puntos que tiene asignados.
4. Repetir 2–3 hasta que los centros dejen de moverse.

El algoritmo NO sabe cuántos grupos hay (eso lo decide el usuario con K), ni cuáles son. Solo busca dónde poner K centros para que cada punto quede cerca del más próximo.

In [ ]:
# Demo — el profesor ejecuta y explica
# Inicialización deliberadamente mala: los 3 centroides apilados cerca de una esquina.
init_malo = np.array([[7, 3], [7.5, 3], [7, 3.5]])

plt.figure()
plt.scatter(
    df_clientes['gasto_mensual_miles'],
    df_clientes['visitas_por_mes'],
    s=20, color='gray', alpha=0.7,
)
plt.scatter(
    init_malo[:, 0], init_malo[:, 1],
    marker='X', s=220, color='black', edgecolor='white', linewidth=1.5,
    label='centroides iniciales (mal ubicados)',
)
plt.title('Punto de partida: 3 centroides apilados en una esquina')
plt.xlabel('gasto mensual (miles de pesos)')
plt.ylabel('visitas por mes')
plt.legend(loc='upper left')
plt.show()

In [ ]:
# Demo — el profesor ejecuta y explica
# Frame por frame — los centroides se mueven hacia el centro de su cluster,
# y los puntos cambian de color cuando se reasignan.
iteraciones = [1, 2, 4, 8]
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, max_it in zip(axes.flat, iteraciones):
    km_step = KMeans(
        n_clusters=3,
        init=init_malo,
        n_init=1,
        max_iter=max_it,
        random_state=42,
    ).fit(X_clientes)
    ax.scatter(
        X_clientes[:, 0], X_clientes[:, 1],
        c=km_step.labels_, s=20, cmap='viridis', alpha=0.7,
    )
    ax.scatter(
        km_step.cluster_centers_[:, 0],
        km_step.cluster_centers_[:, 1],
        marker='X', s=220, color='black', edgecolor='white', linewidth=1.5,
    )
    ax.set_title(f'max_iter = {max_it}')
    ax.set_xlabel('gasto mensual (miles)')
    ax.set_ylabel('visitas por mes')

fig.suptitle('K-means: evolución de centroides desde una init mala', y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Demo — el profesor ejecuta y explica
# Resultado final: 3 segmentos de clientes identificados sin haber visto etiquetas.
km_final = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_clientes)

plt.figure()
plt.scatter(
    X_clientes[:, 0], X_clientes[:, 1],
    c=km_final.labels_, s=20, cmap='viridis', alpha=0.7,
)
plt.scatter(
    km_final.cluster_centers_[:, 0],
    km_final.cluster_centers_[:, 1],
    marker='X', s=220, color='black', edgecolor='white', linewidth=1.5,
)
plt.title('K-means convergido (K=3) — 3 segmentos de clientes')
plt.xlabel('gasto mensual (miles de pesos)')
plt.ylabel('visitas por mes')
plt.show()

print('Tamaño de cada segmento:', np.bincount(km_final.labels_))

**Check rápido — K-means es un algoritmo de…**

(a) clasificación supervisada

(b) regresión

(c) clustering, no supervisado

(d) optimización de hiperparámetros

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (c).** K-means agrupa datos SIN etiquetas — descubre estructura, no predice una etiqueta conocida.

</details>

**Pregunta para vos** — sin googlear ni mirar el notebook: ¿qué hace K-means, en una sola frase?

<details>
<summary>Resultado esperado</summary>

Algo en el espíritu de: *"Agrupa puntos parecidos en K grupos descubriendo dónde están los centros"*. Variantes válidas siempre que mencionen "agrupar sin etiquetas" o "encontrar clusters".

</details>

## Feature Scaling en algoritmos de distancia

**Caso concreto:** una inmobiliaria nos pasa una tabla de viviendas con dos columnas: `metros_cuadrados` (entre 50 y 200) y `cantidad_ambientes` (entre 1 y 5). Quieren agrupar las propiedades en tipologías para sus folletos. Las dos columnas tienen escalas muy distintas — una es decenas/cientos, la otra es unidades.

**Problema:** K-means calcula distancias entre puntos. Una diferencia de 50 metros² pesa muchísimo más que una diferencia de 2 ambientes en el cálculo de distancia, aunque para el negocio una diferencia de 2 ambientes sea súper informativa. Resultado: el algoritmo va a agrupar las viviendas casi exclusivamente por su superficie, ignorando la cantidad de ambientes.

Vamos a hacer el experimento dos veces — sin escalar y escalando — y ver cómo cambia la respuesta. Igual que en la lección anterior, la herramienta es `StandardScaler`, pero esta vez la justificación es **distancias dominadas**, no regularización.

> **Definición — feature scaling.** Transformar las columnas numéricas para que tengan escalas comparables. Hay varios tipos, todos resuelven el mismo problema (escalas dispares) con fórmulas distintas:
>
> - **Normalización (min-max scaling):** lleva cada columna al rango `[0, 1]` (o `[-1, 1]`) usando los valores mínimo y máximo de la columna.
>
> - **Estandarización (`StandardScaler`):** centra cada columna en media 0 y la escala para que tenga desvío estándar 1. Es la que vamos a usar en este curso.
>
> Cuando en este notebook hablamos genéricamente de "escalar las features" o "hacer feature scaling", nos referimos a aplicar cualquiera de estos métodos — en nuestro caso, siempre `StandardScaler`.

In [ ]:
# Demo — el profesor ejecuta y explica
# 3 tipologías de viviendas. Diseñamos los datos para que en 'metros_cuadrados'
# los grupos SE SOLAPEN (mismo centro, mucho ruido), y se distingan solo en 'cantidad_ambientes'.
# 'cantidad_ambientes' es entera por naturaleza — un departamento tiene 2 o 3 ambientes, no 2.37.
rng = np.random.default_rng(42)
n_por_grupo = 100
ambientes = np.concatenate([
    np.full(n_por_grupo, 2, dtype=int),  # tipología A: 2 ambientes
    np.full(n_por_grupo, 3, dtype=int),  # tipología B: 3 ambientes
    np.full(n_por_grupo, 4, dtype=int),  # tipología C: 4 ambientes
])
metros = rng.normal(110, 35, 3 * n_por_grupo)  # mismo centro, mucho ruido — los 3 grupos se solapan en m²

df_viv = pd.DataFrame({
    'cantidad_ambientes': ambientes,
    'metros_cuadrados': metros,
})
X_viv = df_viv.to_numpy(dtype=float)
print('Resumen del dataset de viviendas:')
print(df_viv.describe().round(2))

# Plot 1: con escalas auto (lo que matplotlib elige por defecto).
# Cada eje se reescala a su rango — 'parece' que los datos están bien distribuidos.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_viv[:, 0], X_viv[:, 1], s=20, color='gray', alpha=0.7)
axes[0].set_title('Vista con ejes auto-escalados (engañoso)')
axes[0].set_xlabel('cantidad_ambientes')
axes[0].set_ylabel('metros_cuadrados')

# Plot 2: forzamos que ambos ejes tengan el MISMO rango (el de y).
# Así se ve la diferencia REAL de escala — los puntos quedan apilados en una franja vertical chiquita.
axes[1].scatter(X_viv[:, 0], X_viv[:, 1], s=20, color='gray', alpha=0.7)
lo, hi = X_viv[:, 1].min() - 10, X_viv[:, 1].max() + 10
axes[1].set_xlim(0, hi)
axes[1].set_ylim(0, hi)
axes[1].set_title('Mismo rango en ambos ejes — la escala REAL')
axes[1].set_xlabel('cantidad_ambientes')
axes[1].set_ylabel('metros_cuadrados')

plt.tight_layout()
plt.show()

print('\nLeer el plot derecho: los puntos están dispersos en x (ambientes 2–4),')
print('pero ese rango es minúsculo comparado con la dispersión de y (m²),')
print('así que los puntos parecen una raya vertical. Esa relación de escalas es la que ve K-means.')

In [ ]:
# Demo — el profesor ejecuta y explica
# K-means SIN feature scaling — la distancia está dominada por metros_cuadrados,
# así que los clusters quedan como bandas horizontales, NO por tipología.
km_viv = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_viv)

plt.figure()
plt.scatter(X_viv[:, 0], X_viv[:, 1], c=km_viv.labels_, s=20, cmap='viridis', alpha=0.7)
plt.scatter(
    km_viv.cluster_centers_[:, 0],
    km_viv.cluster_centers_[:, 1],
    marker='X', s=220, color='black', edgecolor='white', linewidth=1.5,
)
plt.title('K-means SIN feature scaling — clusters por metros², ignora ambientes')
plt.xlabel('cantidad_ambientes')
plt.ylabel('metros_cuadrados')
plt.show()

print('Notá cómo los 3 clusters son BANDAS HORIZONTALES de m².')
print('La verdadera estructura (3 tipologías por cantidad de ambientes) quedó perdida.')

**Pause and predict** — antes de seguir, arriesgá:

Si aplico feature scaling a todas las features y vuelvo a correr K-means, ¿qué espero ver en el plot?

(a) los mismos clusters

(b) clusters distintos que reflejan ambas variables — y en este caso, separados por cantidad de ambientes

(c) no se forman clusters porque los datos quedan "achatados"

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (b).** Después de escalar, ambas variables aportan equivalentemente al cálculo de distancias. Como la estructura real de los grupos vive en `cantidad_ambientes`, el K-means recupera las 3 tipologías separadas verticalmente.

</details>

In [ ]:
# Demo — el profesor ejecuta y explica
# StandardScaler centra cada columna en 0 y la escala para que tenga desviación estándar 1.
scaler_viv = StandardScaler()
X_viv_scaled = scaler_viv.fit_transform(X_viv)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(X_viv[:, 1], bins=30, color='steelblue', alpha=0.8)
axes[0].set_title('metros_cuadrados ANTES de escalar')
axes[0].set_xlabel('valor')
axes[0].set_ylabel('frecuencia')

axes[1].hist(X_viv_scaled[:, 1], bins=30, color='seagreen', alpha=0.8)
axes[1].set_title('metros_cuadrados DESPUÉS de escalar')
axes[1].set_xlabel('valor (escala estándar)')
axes[1].set_ylabel('frecuencia')

plt.tight_layout()
plt.show()

print('Media por columna después de escalar (≈ 0):', X_viv_scaled.mean(axis=0).round(3))
print('Desvío por columna después de escalar (≈ 1):', X_viv_scaled.std(axis=0).round(3))

In [ ]:
# Demo — el profesor ejecuta y explica
# K-means con feature scaling — ahora ambas variables pesan igual y el algoritmo recupera las 3 tipologías.
km_viv_scaled = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_viv_scaled)

plt.figure()
plt.scatter(
    X_viv_scaled[:, 0], X_viv_scaled[:, 1],
    c=km_viv_scaled.labels_, s=20, cmap='viridis', alpha=0.7,
)
plt.scatter(
    km_viv_scaled.cluster_centers_[:, 0],
    km_viv_scaled.cluster_centers_[:, 1],
    marker='X', s=220, color='black', edgecolor='white', linewidth=1.5,
)
plt.title('K-means CON feature scaling — comienza a considerar los ambientes')
plt.xlabel('cantidad_ambientes (escalada)')
plt.ylabel('metros_cuadrados (escalada)')
plt.show()

print('Notá los 3 clusters ahora considerando los ambientes.')

**Take-away — ¿cuándo aplicar feature scaling?**

Escalar es barato y casi nunca hace daño. Razones para aplicar feature scaling que vas a encontrar en la práctica:

- **Convergencia más rápida** en algoritmos basados en gradiente (regresión logística, redes neuronales).
- **Estabilidad numérica** en modelos sensibles a la magnitud.
- **Comparabilidad de coeficientes** entre features cuando interpretás un modelo lineal.
- **Requisito explícito** de algunas implementaciones.

## Elección de K + elbow method

**Caso concreto:** una cadena de cafeterías quiere clasificar sus 400 locales en "tipos de tienda" (premium, barrio, oficina, etc.) según `ticket_promedio` y `clientes_por_dia`. No saben cuántos tipos hay — ni 3, ni 4, ni 8 — y necesitan elegirlo a partir de los datos.

Hasta acá fijamos K=3 porque *se veía*. ¿Cómo lo elegirías si no supieras cuántos grupos hay? Una heurística clásica es el **elbow method**: graficar la **inercia** (suma de distancias al cuadrado de cada punto a su centroide) en función de K. La inercia siempre baja si subimos K, pero al pasar el K "verdadero" la mejora se vuelve marginal — eso es el **codo**.

In [ ]:
# Demo — el profesor ejecuta y explica
# 400 cafeterías. ¿Cuántos tipos de tienda hay? Mirá rápido y arriesgá un número.
X_cafe, _ = make_blobs(
    n_samples=400,
    centers=[(8, 80), (5, 250), (15, 150), (20, 350)],
    cluster_std=1.5,
    random_state=42,
)
# Ajustamos el segundo eje a una escala 'clientes por día'
X_cafe[:, 1] = X_cafe[:, 1].clip(min=20)
df_cafe = pd.DataFrame(X_cafe, columns=['ticket_promedio_USD', 'clientes_por_dia'])

# Aplicamos feature scaling antes de clusterizar (ya sabemos por qué).
scaler_cafe = StandardScaler()
X_cafe_s = scaler_cafe.fit_transform(X_cafe)

plt.figure()
plt.scatter(X_cafe[:, 0], X_cafe[:, 1], s=20, color='gray', alpha=0.7)
plt.title('400 cafeterías sin colorear — ¿cuántos tipos de tienda ves?')
plt.xlabel('ticket promedio (USD)')
plt.ylabel('clientes por día')
plt.show()

In [ ]:
# Demo — el profesor ejecuta y explica
# Cada punto es la inercia para un valor de K — buscamos el 'codo'.
ks = list(range(1, 9))
inercias = []
for k in ks:
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_cafe_s)
    inercias.append(km_k.inertia_)

plt.figure()
plt.plot(ks, inercias, marker='o')
plt.title('Elbow method: inercia vs K')
plt.xlabel('K (número de clusters)')
plt.ylabel('inercia (suma de distancias² a centroides)')
plt.xticks(ks)
plt.show()

**Check de comprensión** — mirando el plot, ¿qué K elegirías?

(a) K=2

(b) K=4

(c) K=8

(d) cuanto mayor K, mejor

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (b).** El codo está en K=4 — pasar de K=3 a K=4 mejora bastante, pasar de K=4 a K=5 ya casi no mejora.

Sobre la opción **(d)**: es la trampa. Con K=N (un cluster por punto) la inercia sería 0, pero el modelo no agruparía nada — cada punto sería su propio cluster. La elección de K busca un balance entre **explicar bien los datos** y **mantener pocos grupos** (para que el resultado sea simple, interpretable y útil para tomar decisiones).

</details>

In [ ]:
# Demo — el profesor ejecuta y explica
# Confirmación visual: el codo del plot anterior coincide con la estructura real del dataset
km_cafe = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_cafe_s)

plt.figure()
plt.scatter(X_cafe[:, 0], X_cafe[:, 1], c=km_cafe.labels_, s=20, cmap='viridis', alpha=0.7)
plt.title('K-means con K=4 — confirma lo que sugería el codo')
plt.xlabel('ticket promedio (USD)')
plt.ylabel('clientes por día')
plt.show()

**¿Y si elegimos K más grande que el codo?** El modelo igual corre — pero parte clusters reales en sub-grupos artificiales. Comparemos K=4 (lo que sugiere el codo) contra K=6 y K=8:

In [ ]:
# Demo — el profesor ejecuta y explica
# Side-by-side: K=4 (sweet spot) vs K=6 y K=8 (sobre-fragmentación).
ks_compare = [4, 6, 8]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, k in zip(axes, ks_compare):
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_cafe_s)
    ax.scatter(X_cafe[:, 0], X_cafe[:, 1], c=km_k.labels_, s=20, cmap='viridis', alpha=0.7)
    ax.set_title(f'K = {k}   |   inercia = {km_k.inertia_:.1f}')
    ax.set_xlabel('ticket promedio (USD)')
axes[0].set_ylabel('clientes por día')

fig.suptitle('Subir K más allá del codo: empieza a partir clusters reales', y=1.02)
plt.tight_layout()
plt.show()

**Take-away:** el elbow method es una **heurística** — ayuda a elegir K, no lo demuestra. Existen otras métricas (silhouette score, gap statistic) para los casos en que el codo no es claro.

**Regla práctica:** si el codo es claro, confiá. Si no es claro, K es una **decisión de negocio** — depende de cuántos grupos te sirvan accionablemente (si vas a armar 4 campañas de marketing, K=4; si tu equipo solo puede gestionar 3 tipologías, K=3).

## Pipeline de clustering: los 7 pasos

**Caso concreto:** una app de fitness quiere segmentar a sus usuarios según 3 métricas de uso — `sesiones_por_semana`, `minutos_por_sesion` y `features_distintas_usadas` — para personalizar notificaciones. No tienen segmentos pre-definidos.

El ciclo de un proyecto de ML tiene **7 pasos**:

1. **Recogida de datos**.
2. **Preprocesamiento de datos**.
3. **Elegir el modelo adecuado**.
4. **Entrenamiento del modelo**.
5. **Evaluar el modelo**.
6. **Ajuste y optimización de hiperparámetros**.
7. **Predicciones e implementación**.

Vamos a ver los 7 pasos **en una sola celda**, igual que en el pipeline supervisado, para tener todo el flujo a la vista. Los pasos 1, 2, 3, 4 y 5 se ejecutan en código; los pasos 6 y 7 quedan marcados como comentarios para indicar dónde encajarían.

In [ ]:
# Pipeline de clustering mínimo viable: los 7 pasos teóricos en una sola celda.

# PASO 1 — Recogida de datos.
# (Acá generamos un dataset sintético; en producción vendría de la base de datos de la app.)
centros_app = [(2, 8, 3), (5, 25, 6), (10, 45, 12), (15, 60, 18)]
X_app, _ = make_blobs(
    n_samples=500, n_features=3, centers=centros_app, random_state=7, cluster_std=1.5,
)
X_app = np.clip(X_app, 0, None)  # las métricas de uso no pueden ser negativas
df_app = pd.DataFrame(X_app, columns=['sesiones_por_semana', 'minutos_por_sesion', 'features_usadas'])
print('Primeras filas:')
print(df_app.head().round(2))

# PASO 2 — Preprocesamiento de datos.
# OJO con la diferencia clave respecto a supervisado: en clustering NO hay split train/test,
# porque no hay etiquetas que reservar para evaluar. El preprocesamiento clave acá es feature scaling.
scaler_app = StandardScaler()
X_app_scaled = scaler_app.fit_transform(X_app)

# PASO 3 — Elegir el modelo adecuado.
# K-means: simple, rápido, ampliamente entendido. Para la elección de K, en un proyecto real
# haríamos elbow sobre estos datos como paso previo. Acá fijamos K=4 sabiendo cómo construimos
# el dataset (4 grupos sintéticos), para mantener el pipeline corto.
model_app = KMeans(n_clusters=4, random_state=42, n_init=10)

# PASO 4 — Entrenamiento del modelo.
model_app.fit(X_app_scaled)

# PASO 5 — Evaluar el modelo.
# Sin etiquetas verdaderas, evaluamos por inercia (cuán compactos son los clusters)
# y por inspección del tamaño de cada grupo. NO hay accuracy ni F1 acá.
print(f'\nK elegido: {model_app.n_clusters}')
print(f'Inercia: {model_app.inertia_:.2f}')
print(f'Tamaño de cada cluster: {np.bincount(model_app.labels_)}')

# PASO 6 — Ajuste y optimización de hiperparámetros: probaríamos otros K, comparando inercia y silhouette,
# y elegiríamos el que mejor balancee compactness con interpretabilidad de negocio. (omitido en este pipeline mínimo)

# PASO 7 — Predicciones e implementación: aplicar `model_app.predict(X_nuevo_scaled)` a usuarios nuevos
# para asignarles un segmento, y consumir esos segmentos desde el sistema de notificaciones. (omitido)

**Mini-ejercicio** — si recordás el pipeline supervisado de lecciones anteriores, ¿qué etapa(s) cambiaron de forma en este pipeline de clustering?

(a) la carga de datos

(b) el train/test split y la evaluación

(c) el feature scaling

(d) ninguna

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (b).**

- **Train/test split desaparece**: no hay etiquetas que reservar para evaluar contra ellas.
- **La evaluación cambia de forma**: sin etiqueta verdadera no podemos calcular accuracy / F1 / RMSE; usamos métricas internas como **inercia** (compactness de los clusters) y **silhouette**, más **inspección visual** y validación con el equipo de negocio.

Las otras etapas (carga, feature scaling, elección del modelo, entrenamiento, hiperparámetros, despliegue) son estructuralmente iguales que en supervisado.

</details>

**Take-away final:** el pipeline de ML es **el mismo mapa** para los dos paradigmas. La diferencia central está en la etapa de **evaluación**:

- En **supervisado**, comparás predicciones contra etiquetas conocidas — tenés un termómetro claro (accuracy, F1, RMSE).
- En **no supervisado**, no hay etiqueta verdadera — evaluás con métricas internas (inercia, silhouette) y con **interpretación**.

Cuando se puede, muchos proyectos prefieren supervisado: tienen una vara concreta para medir si el modelo funciona o no. Pero hay un montón de problemas reales donde directamente no hay etiquetas — y ahí clustering es la herramienta natural.

## Bonus: ¿qué es un token?

Todo lo que vimos hoy trabajaba con **tablas de números**. Pero el aprendizaje no supervisado también se aplica a texto — y los modelos de ML siempre reciben **números**, nunca letras.

Para alimentar texto a un modelo, primero se lo parte en pedacitos llamados **tokens** (palabras enteras, sub-palabras, o incluso letras), y a cada token se le asigna un número entero según un diccionario predefinido.

Ejemplo intuitivo: la frase `"hola mundo"` podría tokenizarse como `["hola", " mundo"]` y convertirse en `[1052, 8721]`. El modelo nunca ve la palabra "hola" — ve el número `1052`.

In [ ]:
# Demo — tokenización de juguete: cada palabra → un número.
# Los tokenizadores reales usan vocabularios de decenas de miles de tokens y dividen sub-palabras,
# pero la idea central es la misma que esto.
vocab = {'hola': 1, 'mundo': 2, 'clase': 3, 'de': 4, 'ia': 5}
frase = 'hola clase de ia'
tokens = [vocab.get(p, 0) for p in frase.split()]

print(f'Frase original: {frase!r}')
print(f'Tokens (lo que ve el modelo): {tokens}')

Eso es todo lo que necesitás saber por ahora — la idea central es que **texto se transforma en números antes de que cualquier modelo lo procese**. Sin esa traducción, no hay ML que valga sobre lenguaje natural.

## Resumen de la lección

**K-means**
- Algoritmo de clustering. Aprendizaje no supervisado (sin etiquetas "y")
- Es iterativo: elegís K centros → asignás cada punto al centro más cercano → movés cada centro al promedio de sus puntos → repetís hasta que dejen de moverse.
- El **usuario decide K**. El algoritmo no lo descubre solo.
- Es sensible a la **inicialización**. `n_init` corre el algoritmo varias veces con inits distintas y se queda con la mejor. Por eso casi siempre se usa con `n_init` alto.

**Feature scaling en algoritmos de distancia**
- Cualquier algoritmo basado en **distancia** queda **dominado por la feature de mayor escala** si no escalás. Una columna en miles aplasta a una columna en unidades.
- **`StandardScaler`** centra cada columna en media 0 y desvío 1. Es la opción default del curso.
- **Normalization** es un tipo de feature scaling.
- Escalar es **barato y casi nunca hace daño**. Además de resolver distancias dominadas, ayuda a la convergencia (gradiente), a la estabilidad numérica y a comparar coeficientes en modelos lineales.

**Elección de K — elbow method**
- Graficá la **inercia** (suma de distancias² de cada punto a su centroide) en función de K. El **codo** es donde la mejora se vuelve marginal.
- Es una **heurística**, no una demostración. Si el codo no es claro, K se elige por **negocio** — cuántos grupos podés accionar.
- Subir K más allá del codo NO mejora el modelo: empieza a partir clusters reales en sub-grupos artificiales. Con K=N (un cluster por punto) la inercia es 0 pero no agrupa nada.

**Pipeline de clustering vs. supervisado**
- Los **7 pasos del ciclo de ML son los mismos** en ambos paradigmas.
- Las dos diferencias clave en clustering:
  - **No hay train/test split** — no hay etiquetas que reservar.
  - **La evaluación cambia de forma**: sin etiqueta verdadera no hay accuracy / F1 / RMSE. Se evalúa con métricas internas (**inercia**, **silhouette**), inspección visual y validación con el equipo de negocio.

**Tokens**
- Los modelos siempre reciben **números**, nunca letras.
- **Tokenizar** es partir el texto en pedacitos (palabras, sub-palabras, letras) y mapear cada uno a un entero según un diccionario predefinido. El modelo procesa la secuencia de enteros, no el texto.